In [1]:
import os
import sys
from pathlib import Path
import orbax.checkpoint as ocp
import jax
import jax.numpy as jnp
import numpy as np

# Auto-locate checkpoint-61036 across possible Kaggle mount paths
search_roots = ["/kaggle/input", "/kaggle/working"]
ckpt_dir = None

for root in search_roots:
    for path in Path(root).rglob("checkpoint-61036"):
        if path.is_dir():
            ckpt_dir = path
            break
    if ckpt_dir:
        break

if not ckpt_dir:
    raise FileNotFoundError("Could not locate checkpoint-61036 directory automatically.")

print(f"✅ Found checkpoint at: {ckpt_dir}")
print(f"JAX Devices: {jax.devices()}")

✅ Found checkpoint at: /kaggle/input/datasets/sardarkhanyadav/checkpoint-train-session-2/QaptaanLM-0.75B/checkpoints/jax_cpt/checkpoint-61036
JAX Devices: [CpuDevice(id=0)]


In [2]:
# Check raw Orbax metadata files
metadata_files = [
    ckpt_dir / "metadata.json",
    ckpt_dir / "state" / "_CHECKPOINT_METADATA",
    ckpt_dir / "state" / "_METADATA",
]

for p in metadata_files:
    print("=" * 60)
    print(f"File: {p}")
    print("=" * 60)
    if p.exists():
        try:
            content = p.read_text(errors="ignore")
            print(content[:2000] if len(content) > 2000 else content)
        except Exception as e:
            print(f"Error reading file: {e}")
    else:
        print("NOT FOUND")

File: /kaggle/input/datasets/sardarkhanyadav/checkpoint-train-session-2/QaptaanLM-0.75B/checkpoints/jax_cpt/checkpoint-61036/metadata.json
{
  "step": 61036,
  "metadata": {
    "loss": 2.441319227218628,
    "tokens_trained": 999997440,
    "step": 61036
  }
}
File: /kaggle/input/datasets/sardarkhanyadav/checkpoint-train-session-2/QaptaanLM-0.75B/checkpoints/jax_cpt/checkpoint-61036/state/_CHECKPOINT_METADATA
{"item_handlers": "orbax.checkpoint._src.handlers.standard_checkpoint_handler.StandardCheckpointHandler", "metrics": {}, "performance_metrics": {}, "init_timestamp_nsecs": 1787421967691371951, "commit_timestamp_nsecs": 1787422263476848974, "custom_metadata": {}}
File: /kaggle/input/datasets/sardarkhanyadav/checkpoint-train-session-2/QaptaanLM-0.75B/checkpoints/jax_cpt/checkpoint-61036/state/_METADATA
{"tree_metadata": {"('step',)": {"key_metadata": [{"key": "step", "key_type": 2}], "value_metadata": {"value_type": "np.ndarray", "skip_deserialize": false}}, "('params', 'model', 'e

In [3]:
import os
from pathlib import Path

# Print all files and subfolders under the checkpoint directory
print(f"Inspecting: {ckpt_dir}")
for path in sorted(Path(ckpt_dir).rglob("*")):
    if path.is_file():
        rel = path.relative_to(ckpt_dir)
        print(f"  📄 {rel} ({path.stat().st_size} bytes)")
    elif path.is_dir():
        rel = path.relative_to(ckpt_dir)
        print(f"📁 {rel}/")

Inspecting: /kaggle/input/datasets/sardarkhanyadav/checkpoint-train-session-2/QaptaanLM-0.75B/checkpoints/jax_cpt/checkpoint-61036
  📄 metadata.json (122 bytes)
📁 state/
  📄 state/_CHECKPOINT_METADATA (262 bytes)
  📄 state/_METADATA (420533 bytes)
📁 state/d/
  📄 state/d/8f522e45a9b5430b83819070a28c598a (40523 bytes)
  📄 state/manifest.ocdbt (120 bytes)
📁 state/ocdbt.process_0/
📁 state/ocdbt.process_0/d/
  📄 state/ocdbt.process_0/d/0470ab2436f124858da2ad8998d52952 (49692672 bytes)
  📄 state/ocdbt.process_0/d/047ca18e543510939b32e1367931a556 (1209 bytes)
  📄 state/ocdbt.process_0/d/515435eebae836fa19c8f9ce1e630f49 (2149933056 bytes)
  📄 state/ocdbt.process_0/d/7352442828fbd6271599dc5d3c6813a4 (1315422208 bytes)
  📄 state/ocdbt.process_0/d/db997305f035dab16e06c205c8bd9868 (171 bytes)
  📄 state/ocdbt.process_0/manifest.ocdbt (264 bytes)


In [4]:
import orbax.checkpoint as ocp
from pathlib import Path

ckpt_path = Path("/kaggle/input/datasets/sardarkhanyadav/checkpoint-train-session-2/QaptaanLM-0.75B/checkpoints/jax_cpt/checkpoint-61036")
state_path = ckpt_path / "state"

restored = None

# Option 1: Standard restore targeting state/ directly
try:
    print("Trying StandardCheckpointer on state/ ...")
    checkpointer = ocp.StandardCheckpointer()
    restored = checkpointer.restore(state_path)
    print("✅ Success via StandardCheckpointer on state/")
except Exception as e:
    print(f"Option 1 failed: {e}")

# Option 2: Composite / CheckpointArgs restore targeting root
if restored is None:
    try:
        print("\nTrying Composite restore with item='state' ...")
        checkpointer = ocp.Checkpointer(ocp.StandardCheckpointHandler())
        args = ocp.args.StandardRestore()
        restored = checkpointer.restore(state_path, args=args)
        print("✅ Success via StandardCheckpointHandler!")
    except Exception as e:
        print(f"Option 2 failed: {e}")

# Option 3: PyTreeCheckpointHandler directly on state/
if restored is None:
    try:
        print("\nTrying PyTreeCheckpointHandler on state/ ...")
        handler = ocp.PyTreeCheckpointHandler()
        checkpointer = ocp.Checkpointer(handler)
        restored = checkpointer.restore(state_path)
        print("✅ Success via PyTreeCheckpointHandler on state/")
    except Exception as e:
        print(f"Option 3 failed: {e}")

# Option 4: Full CheckpointManager on parent directory
if restored is None:
    try:
        print("\nTrying CheckpointManager on jax_cpt/ parent ...")
        parent_dir = ckpt_path.parent
        # ItemNames format (state)
        mngr = ocp.CheckpointManager(
            parent_dir,
            item_names=('state',),
        )
        restored_mngr = mngr.restore(61036)
        restored = restored_mngr.state if hasattr(restored_mngr, 'state') else restored_mngr
        print("✅ Success via CheckpointManager!")
    except Exception as e:
        print(f"Option 4 failed: {e}")

print(f"\nResult: {'SUCCESS' if restored is not None else 'FAILED'}")
if restored is not None:
    print(f"Type of restored object: {type(restored)}")

Trying StandardCheckpointer on state/ ...
✅ Success via StandardCheckpointer on state/

Result: SUCCESS
Type of restored object: <class 'dict'>


In [5]:
def inspect_tree(tree, prefix=""):
    if isinstance(tree, dict):
        for k, v in tree.items():
            if isinstance(v, dict):
                print(f"{prefix}📁 {k}/")
                inspect_tree(v, prefix + "  ")
            elif hasattr(v, "shape"):
                print(f"{prefix}📄 {k}: shape={v.shape}, dtype={v.dtype}")
            else:
                print(f"{prefix}📄 {k}: {type(v)}")
    elif hasattr(tree, "__dict__"):
        for k, v in vars(tree).items():
            if not k.startswith("_"):
                print(f"{prefix}📁 {k}/")
                inspect_tree(v, prefix + "  ")
    else:
        if hasattr(tree, "shape"):
            print(f"{prefix}📄 shape={tree.shape}, dtype={tree.dtype}")

print("=== Parameter Hierarchy ===")
inspect_tree(restored)

=== Parameter Hierarchy ===
📄 opt_state: <class 'list'>
📁 params/
  📁 model/
    📁 embed_tokens/
      📄 embedding: shape=(248320, 1024), dtype=bfloat16
    📁 layers_0/
      📁 input_layernorm/
        📄 weight: shape=(1024,), dtype=bfloat16
      📁 linear_attn/
        📄 A_log: shape=(16,), dtype=bfloat16
        📄 conv1d_weight: shape=(6144, 1, 4), dtype=bfloat16
        📄 dt_bias: shape=(16,), dtype=bfloat16
        📁 in_proj_a/
          📄 kernel: shape=(1024, 16), dtype=bfloat16
        📁 in_proj_b/
          📄 kernel: shape=(1024, 16), dtype=bfloat16
        📁 in_proj_qkv/
          📄 kernel: shape=(1024, 6144), dtype=bfloat16
        📁 in_proj_z/
          📄 kernel: shape=(1024, 2048), dtype=bfloat16
        📁 norm/
          📄 weight: shape=(128,), dtype=bfloat16
        📁 out_proj/
          📄 kernel: shape=(2048, 1024), dtype=bfloat16
      📁 mlp/
        📁 down_proj/
          📄 kernel: shape=(3584, 1024), dtype=bfloat16
        📁 gate_proj/
          📄 kernel: shape=(1024, 

In [7]:
def check_weight_health(tree, name=""):
    """Recursively checks for NaN, Inf, and logs L2 norm / mean / std."""
    if isinstance(tree, dict):
        for k, v in tree.items():
            check_weight_health(v, f"{name}.{k}" if name else k)
    elif hasattr(tree, "__dict__"):
        for k, v in vars(tree).items():
            if not k.startswith("_"):
                check_weight_health(v, f"{name}.{k}" if name else k)
    else:
        if hasattr(tree, "shape") and tree.shape != ():
            arr = np.asarray(tree)
            has_nan = np.isnan(arr).any()
            has_inf = np.isinf(arr).any()
            mean_val = float(np.mean(arr))
            std_val = float(np.std(arr))
            norm_val = float(np.linalg.norm(arr))
            
            status = "❌ CORRUPTED" if (has_nan or has_inf) else "✅ OK"
            print(f"{status} | {name:<50} | mean: {mean_val:+.4e} | std: {std_val:.4e} | norm: {norm_val:.4e}")

print("=== Parameter Health Check ===")
# Isolate 'params' or 'target' if it is wrapped in TrainState
params_to_check = restored.get("params", restored) if isinstance(restored, dict) else getattr(restored, "params", restored)
check_weight_health(params_to_check)

=== Parameter Health Check ===
✅ OK | model.embed_tokens.embedding                       | mean: +1.2573e-07 | std: 1.8024e-04 | norm: 3.1663e+02
✅ OK | model.layers_0.input_layernorm.weight              | mean: +1.2109e-01 | std: 1.4941e-01 | norm: 8.5649e+00
✅ OK | model.layers_0.linear_attn.A_log                   | mean: -1.9766e+00 | std: 1.4453e+00 | norm: 9.8067e+00
✅ OK | model.layers_0.linear_attn.conv1d_weight           | mean: +6.2943e-04 | std: 5.1025e-02 | norm: 1.1441e+01
✅ OK | model.layers_0.linear_attn.dt_bias                 | mean: -1.9062e+00 | std: 5.7500e+00 | norm: 2.4264e+01
✅ OK | model.layers_0.linear_attn.in_proj_a.kernel        | mean: -6.3324e-04 | std: 1.7212e-02 | norm: 4.8663e+00
✅ OK | model.layers_0.linear_attn.in_proj_b.kernel        | mean: +3.7003e-04 | std: 8.8501e-03 | norm: 2.2563e+00
✅ OK | model.layers_0.linear_attn.in_proj_qkv.kernel      | mean: -1.0654e-06 | std: 1.6708e-03 | norm: 4.1767e+01
✅ OK | model.layers_0.linear_attn.in_proj_z.kerne

In [8]:
import torch
import numpy as np
from pathlib import Path
from safetensors.torch import load_file

# 1. Load HF safetensors dictionary directly
safetensors_path = Path("/kaggle/input/datasets/sardarkhanyadav/checkpoint-train-session-2/QaptaanLM-0.75B/checkpoints/jax_cpt_hf/model.safetensors")
hf_sd = load_file(str(safetensors_path))
print(f"Loaded {len(hf_sd)} tensors from safetensors.")

# 2. Helpers
def get_jax(path):
    x = restored
    for part in path.split("."):
        x = x[part]
    # Cast bfloat16 to float32 at the numpy level before PyTorch ingestion
    return np.asarray(x).astype(np.float32)

def compare(jax_path, hf_path, transpose=False, squeeze_dim=None):
    jax_arr = get_jax(jax_path)
    jax_t = torch.from_numpy(jax_arr)
    
    if transpose:
        jax_t = jax_t.T
    if squeeze_dim is not None:
        jax_t = jax_t.squeeze(squeeze_dim)
        
    if hf_path not in hf_sd:
        print(f"\n❌ '{hf_path}' not found in safetensors!")
        return

    hf_t = hf_sd[hf_path].float()

    print(f"\n🔍 JAX: {jax_path}  ->  HF: {hf_path}")
    print(f"   JAX shape: {tuple(jax_t.shape)}")
    print(f"   HF  shape: {tuple(hf_t.shape)}")

    if jax_t.shape != hf_t.shape:
        print("   ❌ SHAPE MISMATCH")
        return

    cos = torch.nn.functional.cosine_similarity(
        jax_t.flatten().unsqueeze(0),
        hf_t.flatten().unsqueeze(0)
    ).item()

    max_diff = torch.max(torch.abs(jax_t - hf_t)).item()
    mean_diff = torch.mean(torch.abs(jax_t - hf_t)).item()

    print(f"   Cosine similarity: {cos:.8f}")
    print(f"   Mean abs diff:     {mean_diff:.8e}")
    print(f"   Max abs diff:      {max_diff:.8e}")


# ---------------------------------------------------------
# Test Comparisons (Layer 0)
# ---------------------------------------------------------

# Check shape of Conv1d in safetensors to auto-squeeze if necessary
hf_conv_shape = hf_sd.get("model.layers.0.linear_attn.conv1d.weight", torch.tensor([])).shape
sq_dim = 1 if len(hf_conv_shape) == 2 else None

compare(
    "params.model.layers_0.linear_attn.conv1d_weight",
    "model.layers.0.linear_attn.conv1d.weight",
    squeeze_dim=sq_dim
)

compare(
    "params.model.layers_0.linear_attn.in_proj_qkv.kernel",
    "model.layers.0.linear_attn.in_proj_qkv.weight",
    transpose=True
)

compare(
    "params.model.layers_0.linear_attn.in_proj_z.kernel",
    "model.layers.0.linear_attn.in_proj_z.weight",
    transpose=True
)

compare(
    "params.model.layers_0.linear_attn.in_proj_b.kernel",
    "model.layers.0.linear_attn.in_proj_b.weight",
    transpose=True
)

compare(
    "params.model.layers_0.linear_attn.in_proj_a.kernel",
    "model.layers.0.linear_attn.in_proj_a.weight",
    transpose=True
)

compare(
    "params.model.layers_0.linear_attn.out_proj.kernel",
    "model.layers.0.linear_attn.out_proj.weight",
    transpose=True
)

Loaded 321 tensors from safetensors.

🔍 JAX: params.model.layers_0.linear_attn.conv1d_weight  ->  HF: model.layers.0.linear_attn.conv1d.weight
   JAX shape: (6144, 1, 4)
   HF  shape: (6144, 1, 4)
   Cosine similarity: 0.00178624
   Mean abs diff:     6.03991002e-02
   Max abs diff:      1.77966309e+00

🔍 JAX: params.model.layers_0.linear_attn.in_proj_qkv.kernel  ->  HF: model.layers.0.linear_attn.in_proj_qkv.weight
   JAX shape: (6144, 1024)
   HF  shape: (6144, 1024)
   Cosine similarity: 0.99982691
   Mean abs diff:     0.00000000e+00
   Max abs diff:      0.00000000e+00

🔍 JAX: params.model.layers_0.linear_attn.in_proj_z.kernel  ->  HF: model.layers.0.linear_attn.in_proj_z.weight
   JAX shape: (2048, 1024)
   HF  shape: (2048, 1024)
   Cosine similarity: 0.99977833
   Mean abs diff:     0.00000000e+00
   Max abs diff:      0.00000000e+00

🔍 JAX: params.model.layers_0.linear_attn.in_proj_b.kernel  ->  HF: model.layers.0.linear_attn.in_proj_b.weight
   JAX shape: (16, 1024)
   HF  sh

In [9]:
import numpy as np
import torch
from pathlib import Path
from safetensors.torch import load_file

# Load raw tensors directly from file
safetensors_path = Path("/kaggle/input/datasets/sardarkhanyadav/checkpoint-train-session-2/QaptaanLM-0.75B/checkpoints/jax_cpt_hf/model.safetensors")
hf_sd = load_file(str(safetensors_path))

LINEAR_LAYERS = [
    0, 1, 2,
    4, 5, 6,
    8, 9, 10,
    12, 13, 14,
    16, 17, 18,
    20, 21, 22
]

results = []

for i in LINEAR_LAYERS:
    # Convert numpy bfloat16 -> float32 -> PyTorch Tensor
    jax_arr = np.asarray(
        restored["params"]["model"][f"layers_{i}"]["linear_attn"]["conv1d_weight"]
    ).astype(np.float32)
    jax_tensor = torch.from_numpy(jax_arr)

    hf_key = f"model.layers.{i}.linear_attn.conv1d.weight"
    if hf_key not in hf_sd:
        print(f"Layer {i:2d}: ❌ Key '{hf_key}' not found in safetensors!")
        continue

    hf_tensor = hf_sd[hf_key].float()

    cos = torch.nn.functional.cosine_similarity(
        jax_tensor.flatten().unsqueeze(0),
        hf_tensor.flatten().unsqueeze(0)
    ).item()

    results.append((i, cos))

    print(
        f"Layer {i:2d}: "
        f"JAX shape={tuple(jax_tensor.shape)} | "
        f"HF shape={tuple(hf_tensor.shape)} | "
        f"cos={cos:.8f}"
    )

Layer  0: JAX shape=(6144, 1, 4) | HF shape=(6144, 1, 4) | cos=0.00178624
Layer  1: JAX shape=(6144, 1, 4) | HF shape=(6144, 1, 4) | cos=-0.00072737
Layer  2: JAX shape=(6144, 1, 4) | HF shape=(6144, 1, 4) | cos=0.00482718
Layer  4: JAX shape=(6144, 1, 4) | HF shape=(6144, 1, 4) | cos=-0.00000891
Layer  5: JAX shape=(6144, 1, 4) | HF shape=(6144, 1, 4) | cos=0.00038319
Layer  6: JAX shape=(6144, 1, 4) | HF shape=(6144, 1, 4) | cos=0.00657008
Layer  8: JAX shape=(6144, 1, 4) | HF shape=(6144, 1, 4) | cos=0.00744099
Layer  9: JAX shape=(6144, 1, 4) | HF shape=(6144, 1, 4) | cos=-0.00404184
Layer 10: JAX shape=(6144, 1, 4) | HF shape=(6144, 1, 4) | cos=0.00207885
Layer 12: JAX shape=(6144, 1, 4) | HF shape=(6144, 1, 4) | cos=-0.00167690
Layer 13: JAX shape=(6144, 1, 4) | HF shape=(6144, 1, 4) | cos=-0.00192165
Layer 14: JAX shape=(6144, 1, 4) | HF shape=(6144, 1, 4) | cos=-0.00653882
Layer 16: JAX shape=(6144, 1, 4) | HF shape=(6144, 1, 4) | cos=-0.00307452
Layer 17: JAX shape=(6144, 1, 4

In [10]:
import os
import shutil
from pathlib import Path
import numpy as np
import torch
from safetensors.torch import load_file, save_file

# Paths
source_hf_dir = Path("/kaggle/input/datasets/sardarkhanyadav/checkpoint-train-session-2/QaptaanLM-0.75B/checkpoints/jax_cpt_hf")
output_dir = Path("/kaggle/working/qaptaan_repaired")
output_dir.mkdir(parents=True, exist_ok=True)

# 1. Load the original safetensors dictionary directly into CPU memory
print("Loading original model.safetensors...")
state_dict = load_file(str(source_hf_dir / "model.safetensors"))

LINEAR_LAYERS = [
    0, 1, 2,
    4, 5, 6,
    8, 9, 10,
    12, 13, 14,
    16, 17, 18,
    20, 21, 22
]

# 2. Patch each linear attention layer's conv1d.weight with intact JAX parameters
print("\nPatching conv1d weights from JAX PyTree...")
for i in LINEAR_LAYERS:
    # Extract JAX array and convert bfloat16 -> float32 -> PyTorch Tensor
    jax_conv_np = np.asarray(
        restored["params"]["model"][f"layers_{i}"]["linear_attn"]["conv1d_weight"]
    ).astype(np.float32)
    
    hf_key = f"model.layers.{i}.linear_attn.conv1d.weight"
    
    if hf_key not in state_dict:
        print(f"⚠️ Warning: {hf_key} not in original state_dict. Adding it.")
        target_shape = None
    else:
        target_shape = state_dict[hf_key].shape

    # Handle shape alignment: JAX (6144, 1, 4) vs PyTorch (6144, 1, 4) or (6144, 4)
    if target_shape is not None and len(target_shape) == 2:
        jax_conv_tensor = torch.from_numpy(jax_conv_np).squeeze(1).bfloat16()
    else:
        jax_conv_tensor = torch.from_numpy(jax_conv_np).bfloat16()

    state_dict[hf_key] = jax_conv_tensor
    print(f"✅ Layer {i:2d}: Updated {hf_key} with shape {tuple(jax_conv_tensor.shape)}")

# 3. Save the repaired safetensors file
repaired_safetensors_path = output_dir / "model.safetensors"
print(f"\nSaving repaired safetensors to {repaired_safetensors_path}...")
save_file(state_dict, str(repaired_safetensors_path))
print("✅ Saved repaired weights!")

# 4. Copy config and tokenizer metadata files
for fname in ["config.json", "tokenizer.json", "tokenizer_config.json", "chat_template.jinja"]:
    src_file = source_hf_dir / fname
    if src_file.exists():
        shutil.copy(src_file, output_dir / fname)
        print(f"Copied {fname}")

print(f"\n🎉 Done! Ready-to-upload checkpoint is in: {output_dir}")

Loading original model.safetensors...

Patching conv1d weights from JAX PyTree...
✅ Layer  0: Updated model.layers.0.linear_attn.conv1d.weight with shape (6144, 1, 4)
✅ Layer  1: Updated model.layers.1.linear_attn.conv1d.weight with shape (6144, 1, 4)
✅ Layer  2: Updated model.layers.2.linear_attn.conv1d.weight with shape (6144, 1, 4)
✅ Layer  4: Updated model.layers.4.linear_attn.conv1d.weight with shape (6144, 1, 4)
✅ Layer  5: Updated model.layers.5.linear_attn.conv1d.weight with shape (6144, 1, 4)
✅ Layer  6: Updated model.layers.6.linear_attn.conv1d.weight with shape (6144, 1, 4)
✅ Layer  8: Updated model.layers.8.linear_attn.conv1d.weight with shape (6144, 1, 4)
✅ Layer  9: Updated model.layers.9.linear_attn.conv1d.weight with shape (6144, 1, 4)
✅ Layer 10: Updated model.layers.10.linear_attn.conv1d.weight with shape (6144, 1, 4)
✅ Layer 12: Updated model.layers.12.linear_attn.conv1d.weight with shape (6144, 1, 4)
✅ Layer 13: Updated model.layers.13.linear_attn.conv1d.weight with

In [ ]:
!pip install -U --no-cache-dir git+https://github.com/huggingface/transformers.git

In [ ]:
!pip install -U safetensors

In [11]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

REPAIRED = "/kaggle/working/qaptaan_repaired"

tokenizer = AutoTokenizer.from_pretrained(
    REPAIRED,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    REPAIRED,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

prompts = [
    "def two_sum(nums: list[int], target: int) -> list[int]:\n"
    '    """Return indices of two numbers that add up to target."""\n',

    "def binary_search(arr: list[int], target: int) -> int:\n"
    '    """Return index of target in sorted arr, or -1 if not found."""\n',

    "import numpy as np\n\n"
    "def normalize(x: np.ndarray) -> np.ndarray:\n"
    '    """Normalize an array to zero mean and unit variance."""\n',
]

for i, prompt in enumerate(prompts, 1):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    print("\n" + "=" * 80)
    print(f"TEST {i}")
    print("=" * 80)
    print(text)

[transformers] The tokenizer you are loading from '/kaggle/working/qaptaan_repaired' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/321 [00:00<?, ?it/s]


TEST 1
def two_sum(nums: list[int], target: int) -> list[int]:
    """Return indices of two numbers that add up to target."""
       ORMALdielectricipsowielsichtigtrollerscionadosNW.ops.normalizedzwidedowiels双色filtr NortesterONIowiels.AI.mit!/Users/applicationnji.AI!/booatorios1YWatoriosYWResponses
stownYWResponsesvearinYWResponsesveYWoniiientosientosProtocolズ.AI!/Protocolズustraliacersowie Nimbus INDUST人士的cionaisNWatoriosPROTO1YWResponsesensteiatorsatorios1YW堃 Suites情形之一的ligenシャルلز Suites情形之一的ialis人士謗ינוズieuseAIT Nationalsierungen Suites情形之一的ialis人士謗 셨ishly1YWResponses 셨lessly1YWatoryYWResponses................................................................iskuYWResponses SERIES

TEST 2
def binary_search(arr: list[int], target: int) -> int:
    """Return index of target in sorted arr, or -1 if not found."""
opusizen-windows��whilealus.hits_EQUALSIZEComparatorweitenRIESbleremukсльlichkeitesserONIathanumbnailнетOPSISProducionar complementoicioopatadanJwtelicGeneresteValoratorioBenefICI

In [13]:
import numpy as np
import torch
from safetensors.torch import load_file

LINEAR_LAYERS = [
    0, 1, 2,
    4, 5, 6,
    8, 9, 10,
    12, 13, 14,
    16, 17, 18,
    20, 21, 22
]

path = "/kaggle/working/qaptaan_repaired/model.safetensors"
sd = load_file(path)

print("Post-save verification:")

for i in LINEAR_LAYERS:
    # Cast bfloat16 -> float32 at numpy level
    jax_arr = np.asarray(
        restored["params"]["model"][f"layers_{i}"]["linear_attn"]["conv1d_weight"]
    ).astype(np.float32)

    jax_tensor = torch.from_numpy(jax_arr)
    hf_tensor = sd[f"model.layers.{i}.linear_attn.conv1d.weight"].float()

    cos = torch.nn.functional.cosine_similarity(
        jax_tensor.flatten().unsqueeze(0),
        hf_tensor.flatten().unsqueeze(0)
    ).item()

    print(f"Layer {i:2d}: cosine similarity = {cos:.8f}")

Post-save verification:
Layer  0: cosine similarity = 1.00000155
Layer  1: cosine similarity = 0.99999964
Layer  2: cosine similarity = 0.99999982
Layer  4: cosine similarity = 0.99999917
Layer  5: cosine similarity = 0.99999952
Layer  6: cosine similarity = 0.99999857
Layer  8: cosine similarity = 1.00000024
Layer  9: cosine similarity = 1.00000215
Layer 10: cosine similarity = 1.00000060
Layer 12: cosine similarity = 1.00000024
Layer 13: cosine similarity = 1.00000024
Layer 14: cosine similarity = 1.00000060
Layer 16: cosine similarity = 1.00000107
Layer 17: cosine similarity = 1.00000179
Layer 18: cosine similarity = 1.00000262
Layer 20: cosine similarity = 1.00000167
Layer 21: cosine similarity = 1.00000191
Layer 22: cosine similarity = 1.00000072
